## **Local S3 Setup**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

### **01. S3 Config values**

In [ ]:
USE_S3 = os.getenv("USE_S3", "false").lower() == "true"
S3_BUCKET = os.getenv("S3_BUCKET", "")
MEMORY_DIR = os.getenv("MEMORY_DIR", "../memory")

**os.getenv("USE_S3", "false")** means: “Read the variable called USE_S3; if it does not exist, use the text "false" instead.”

### **02. S3 Config Client**

**What is boto3.client("s3")?**

boto3 is the AWS SDK for Python, and client("s3") creates a low-level S3 service client that can do operations like:
1. upload an object,
2. download an object,
3. check bucket content.

In [ ]:
if USE_S3:
    s3_client = boto3.client("s3")

for example: 

**s3.put_object(Bucket="my-bucket", Key="test.txt", Body="hello")**

*That uploads a file-like object named test.txt into the bucket.*

## **Local Lambda  Handler**

In [ ]:
from mangum import Mangum
from TWIN.server import app

In [ ]:
# Create the Lambda handler
handler = Mangum(app)

### **01. What problem does this solve?**




`server.py` has a FastAPI app. FastAPI is designed as an ASGI web application, which normally runs behind an ASGI server like Uvicorn. `Mangum` acts as a wrapper or adapter so that Lambda events from API Gateway can be translated into ASGI requests and your FastAPI app can handle them.

**Normal local world**

When you run locally:

```python
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

The flow is:

1. Uvicorn starts a web server.
2. A browser sends an HTTP request to that server.
3. Uvicorn passes the request to FastAPI.
4. The FastAPI route runs.
5. The response goes back.

That is the normal web-server model.

**AWS Lambda world**

Lambda does not run a normal always-on web server. Instead:

1. API Gateway receives the HTTP request.
2. It converts that request into an AWS event object.
3. Lambda invokes a function handler with `event` and `context`.
4. That handler must return a Lambda-style response object.

FastAPI by itself does not speak that Lambda event format. So you need a translator.

That translator is `Mangum`.


### **02. What is `handler` here?**

```python
handler = Mangum(app)
```

This creates a new object called `handler` by wrapping your FastAPI app with `Mangum`. `Mangum` exposes a callable Lambda-compatible interface, so AWS Lambda can invoke it as the function entry point. The Mangum adapter defines a `__call__` method, which lets the instance behave like a function handler.

**Very important point**

`handler` is the thing AWS Lambda actually runs.  
Not `app`.  
Not `server.py` directly.  
`handler`.

That is why in Lambda configuration you set something like:

```text
lambda_handler.handler
```

Meaning:

- File name = `lambda_handler.py`
- Variable or function name = `handler`